# UrbanStyle RFM alalüüs

### Grupp: Toode

In [1]:
import pandas as pd
# enne koodi jooksutamist, kontrollida, kas failid on õiges kaustas olemas. 
import os
print("Pythoni praegune asukoht:", os.getcwd())
print("Failid selles kaustas:", os.listdir())
df_customers = pd.read_csv('urbanstyle_customers.csv')
df_sales = pd.read_csv('urbanstyle_sales.csv')



Pythoni praegune asukoht: c:\Users\kertu\Documents\DACA-Python\venv
Failid selles kaustas: ['.gitignore', 'etc', 'Hajuvusdiagramm.png', 'hello.py', 'Include', 'Iseseisev töö.ipynb', 'Kliendisegmendi analüüs.png', 'Lib', 'main.py', 'pyvenv.cfg', 'Scripts', 'share', 'test_notebook.ipynb', 'TOP 10 Klienti.png', 'urbanstyle_customers.csv', 'urbanstyle_rfm_results.csv', 'urbanstyle_sales.csv', 'week7_rfm_complete.ipynb', 'week7_rfm_ROLL A & B.ipynb', 'week7_rfm_rollD.ipynb']


 ## ROLL A - DATA LOADING

In [2]:
#liidame sales ja customers tabelid kokku, et saada juurde kliendi email ja eesnimi.
df = pd.merge(df_sales, df_customers[['customer_id', 'email', 'first_name']], on='customer_id', how='left')

In [3]:
# Laadime sales tabeli DataFrame'i ja kontrollime selle kuju (shape) ning esimeseid ridu (head)
print("Sales shape:", df_sales.shape)
display(df_sales.head())

Sales shape: (15234, 11)


,sale_id,invoice_id,sale_date,customer_id,product_id,quantity,unit_price,total_price,channel,store_location,payment_method
0,1,INV-202301-00001,2023-01-10,2588.0,1274,2,234.79,469.58,pood,Tallinn,kaart
1,2,INV-202301-00002,2023-01-16,4338.0,1207,2,241.13,482.26,pood,Pärnu,järelmaks
2,3,INV-202301-00003,2023-01-05,4673.0,1264,1,258.46,221.19,pood,Pärnu,järelmaks
3,4,INV-202301-00004,2023-01-02,4677.0,1341,3,45.21,135.63,pood,Tartu,sularaha
4,5,INV-202301-00005,2023-01-13,2390.0,1284,1,99.57,99.57,pood,Tartu,kaart


In [4]:
# Laadime customers tabeli DataFrame'i ja kontrollime selle kuju (shape) ning esimeseid ridu (head)
print("Customers shape:", df_customers.shape)
display(df_customers.head())

Customers shape: (3150, 9)


,customer_id,first_name,last_name,email,phone,city,registration_date,loyalty_tier,birth_year
0,2001,Eha,Aas,eha.aas@telia.ee,+372 8713 1455,Tallinn,2024-02-27,NaN,1973
1,2002,Aivar,Kõiv,aivar.koiv@outlook.com,+372 8943 8684,Haapsalu,2025-01-09,bronze,1988
2,2003,Maris,Rebane,maris.rebane@telia.ee,+372 5918 5726,Tartu,2021-02-03,NaN,1999
3,2004,Jaak,Talvik,jaak.talvik@mail.ee,+372 8554 4232,Tallinn,2023-11-12,silver,1974
4,2005,Raivo,Koppel,raivo.koppel@yahoo.com,+372 5298 4365,Tallinn,2023-05-22,bronze,2004


In [5]:
# Tulemuste kontroll
print("Uus shape:", df.shape)
display(df.head())

Uus shape: (15234, 13)


,sale_id,invoice_id,sale_date,customer_id,product_id,quantity,unit_price,total_price,channel,store_location,payment_method,email,first_name
0,1,INV-202301-00001,2023-01-10,2588.0,1274,2,234.79,469.58,pood,Tallinn,kaart,NaN,Hille
1,2,INV-202301-00002,2023-01-16,4338.0,1207,2,241.13,482.26,pood,Pärnu,järelmaks,merle.luik@mail.ee,Merle
2,3,INV-202301-00003,2023-01-05,4673.0,1264,1,258.46,221.19,pood,Pärnu,järelmaks,liina.saar@gmail.com,Liina
3,4,INV-202301-00004,2023-01-02,4677.0,1341,3,45.21,135.63,pood,Tartu,sularaha,aili.pihl@yahoo.com,Aili
4,5,INV-202301-00005,2023-01-13,2390.0,1284,1,99.57,99.57,pood,Tartu,kaart,triin.lill@telia.ee,Triin


## ROLL B - DATA CLEANING

In [6]:
# vaatan mitu rida oli enne duplikaatite eemaldamist, vastuseks tuleb 15234 rida, mis on õige
print("Esialgne shape:", df.shape)

Esialgne shape: (15234, 13)


In [7]:
# Duplikaatide kontroll ja eemaldamine invoice_id järgi
print("Duplikaadid:", df.duplicated(subset=['invoice_id']).sum())
df = df.drop_duplicates(subset=['invoice_id'], keep='first')

Duplikaadid: 5116


In [8]:
#Kontrolli NULL väärtusi:
print("NULL-id:\n", df.isnull().sum())
# Tulemus: Null väärtused on customer_id, store_location, email ja first_name veergudes. 


NULL-id:
 sale_id              0
invoice_id           0
sale_date            0
customer_id        988
product_id           0
quantity             0
unit_price           0
total_price          0
channel              0
store_location    3462
payment_method       0
email             1944
first_name         988
dtype: int64


In [9]:
#Eemaldame need read, kus väärtused on null, kuna see on oluline veerg RFM analüüsi jaoks.
df = df.dropna(subset=['customer_id', 'sale_date', 'total_price'])

In [10]:
#Kuna kuupäeva formaat on erinev, siis selleks, et saaksime jooksutada järgmist koodi on alguses vaja saada formaat õigeks kus päev on esimesena märgitud, näiteks 25/01/2023, Pandas üritab automaatselt kuvada teises vormis
df['sale_date'] = pd.to_datetime(df['sale_date'], format='mixed')

In [11]:
# Konverteerime (parsime) tekstiformaadis (string) kuupäevad pandas'e jaoks õigesse kuupäeva- ja kellaajavormingusse (datetime).  
# Tänu sellele saab Pandas hiljem õigesti aru kalendrilistest kuupäevadest, mis on hädavajalik näiteks RFM-analüüsis Recency (päevade arvu viimasest ostust) arvutamiseks
df['sale_date'] = pd.to_datetime(df['sale_date'])

In [12]:
# Kontrolli outlier'id: kas on negatiivseid total_price väärtusi?
print("Negatiivsed/null väärtused:", (df['total_price'] <= 0).sum())

Negatiivsed/null väärtused: 180


In [13]:
# Eemaldame negatiivsed total_price väärtused, kuna need on ilmselt andmete sisestamise vead ja ei ole mõistlikud RFM analüüsi jaoks.
df = df[df['total_price'] > 0]

In [14]:
# Kontrollime uuesti Otlier'id: kas on negatiivseid total_price väärtusi?
print("Negatiivsed/null väärtused:", (df['total_price'] <= 0).sum())
# Tulemus on 0, mis on õige.

Negatiivsed/null väärtused: 0


In [15]:
print("--- PUHASTUSRAPORT ---")
print("Lõplik shape:", df.shape)        #Pärast duplikaatide, puuduvate väärtuste ja outlier'ite eemaldamist jäi andmestikku alles 13 468 rida ja 13 veergu
print("Unikaalsete klientide arv:", df['customer_id'].nunique())
print("Kuupäevavahemik:", df['sale_date'].min(), "kuni", df['sale_date'].max())
# Tulemus 
# 1. Müügitabeli RAW mahuks oli 15 234 rida (Roll A).
# 2. Pärast müügiarvete deduplitseerimist (invoice_id järgi) jäi alles 10 118 reaalset tehingut.
# 3. Kliendiandmete liitmisel (left join) ja subsequent NULL-väärtuste (puuduv customer_id) 
#    ning negatiivsete hindade eemaldamisel jäi RFM-analüüsiks alles täpselt 13 468 puhast koondrida.

--- PUHASTUSRAPORT ---
Lõplik shape: (8950, 13)
Unikaalsete klientide arv: 2540
Kuupäevavahemik: 2023-01-01 00:00:00 kuni 2026-09-06 00:00:00


## Roll C – RFM-analüüs

Puhastatud müügiandmete põhjal arvutatakse klientide Recency, Frequency ja Monetary näitajad, määratakse RFM-skoorid ning moodustatakse kliendisegmendid.

**Võrdluskuupäev:** 28.02.2025 (vastavalt juhendile).

In [16]:
# RFM-analüüsi võrdluskuupäev vastavalt juhendile
reference_date = pd.Timestamp("2025-02-28")

# Kontrollime ostukuupäevi
print("Võrdluskuupäev:", reference_date.date())
print("Viimane ostukuupäev:", df["sale_date"].max().date())
print("Hilisemate ostude arv:", (df["sale_date"] > reference_date).sum())

Võrdluskuupäev: 2025-02-28
Viimane ostukuupäev: 2026-09-06
Hilisemate ostude arv: 35


In [17]:
# Jätame RFM-analüüsi ainult võrdluskuupäevani toimunud ostud
df_rfm = df.loc[df["sale_date"] <= reference_date].copy()

print("Puhastatud müügiridu kokku:", len(df))
print("RFM-analüüsi müügiridu:", len(df_rfm))
print("Viimane analüüsi kaasatud ost:", df_rfm["sale_date"].max().date())

Puhastatud müügiridu kokku: 8950
RFM-analüüsi müügiridu: 8915
Viimane analüüsi kaasatud ost: 2025-02-28


In [18]:
# Arvutame iga kliendi RFM-näitajad
rfm = df_rfm.groupby("customer_id").agg(
    Last_Purchase=("sale_date", "max"),
    Frequency=("sale_id", "count"),
    Monetary=("total_price", "sum")
).reset_index()

# Recency = päevade arv viimasest ostust
rfm["Recency"] = (reference_date - rfm["Last_Purchase"]).dt.days

# Kuvame esimesed viis klienti
display(rfm.head())
print("RFM-analüüsi klientide arv:", len(rfm))

,customer_id,Last_Purchase,Frequency,Monetary,Recency
0,2001.0,2024-11-29,2,203.92,91
1,2004.0,2024-12-19,2,1198.56,71
2,2005.0,2024-10-03,4,959.60,148
3,2006.0,2023-11-09,1,327.06,477
4,2007.0,2025-01-30,1,318.63,29


RFM-analüüsi klientide arv: 2538


In [19]:
# RFM-skoorid skaalal 1–5

# Recency: väiksem päevade arv = parem skoor
rfm["R_Score"] = pd.qcut(
    rfm["Recency"].rank(method="first"),
    5,
    labels=[5, 4, 3, 2, 1]
).astype(int)

# Frequency: rohkem oste = parem skoor
rfm["F_Score"] = pd.qcut(
    rfm["Frequency"].rank(method="first"),
    5,
    labels=[1, 2, 3, 4, 5]
).astype(int)

# Monetary: suurem kogukulutus = parem skoor
rfm["M_Score"] = pd.qcut(
    rfm["Monetary"].rank(method="first"),
    5,
    labels=[1, 2, 3, 4, 5]
).astype(int)

# RFM koguskoor
rfm["RFM_Score"] = rfm["R_Score"] + rfm["F_Score"] + rfm["M_Score"]

display(rfm.head())
print("RFM-skoori vahemik:", rfm["RFM_Score"].min(), "kuni", rfm["RFM_Score"].max())

,customer_id,Last_Purchase,Frequency,Monetary,Recency,R_Score,F_Score,M_Score,RFM_Score
0,2001.0,2024-11-29,2,203.92,91,4,1,1,6
1,2004.0,2024-12-19,2,1198.56,71,4,1,4,9
2,2005.0,2024-10-03,4,959.60,148,3,4,4,11
3,2006.0,2023-11-09,1,327.06,477,1,1,1,3
4,2007.0,2025-01-30,1,318.63,29,5,1,1,7


RFM-skoori vahemik: 3 kuni 15


In [20]:
# Kliendisegmentide määramine RFM koguskoori alusel
def rfm_segment(score):
    if score >= 13:
        return "VIP Champions"
    elif score >= 10:
        return "Loyal"
    elif score >= 7:
        return "Potential"
    elif score >= 4:
        return "At Risk"
    else:
        return "Lost"

rfm["Segment"] = rfm["RFM_Score"].apply(rfm_segment)

# Kontrollime segmentide jaotust
print("Klientide arv segmentides:")
print(rfm["Segment"].value_counts())

print("\nKliente kokku:", len(rfm))
display(rfm.head())

Klientide arv segmentides:
Segment
Potential        762
Loyal            677
At Risk          529
VIP Champions    453
Lost             117
Name: count, dtype: int64

Kliente kokku: 2538


,customer_id,Last_Purchase,Frequency,Monetary,Recency,R_Score,F_Score,M_Score,RFM_Score,Segment
0,2001.0,2024-11-29,2,203.92,91,4,1,1,6,At Risk
1,2004.0,2024-12-19,2,1198.56,71,4,1,4,9,Potential
2,2005.0,2024-10-03,4,959.60,148,3,4,4,11,Loyal
3,2006.0,2023-11-09,1,327.06,477,1,1,1,3,Lost
4,2007.0,2025-01-30,1,318.63,29,5,1,1,7,Potential


In [21]:
# Kontrollime RFM-analüüsi lõpptulemust
print("Kliente kokku:", len(rfm))
print("Puuduvaid segmente:", rfm["Segment"].isna().sum())
print("Negatiivseid Recency väärtusi:", (rfm["Recency"] < 0).sum())
print("RFM-skoori vahemik:", rfm["RFM_Score"].min(), "-", rfm["RFM_Score"].max())

Kliente kokku: 2538
Puuduvaid segmente: 0
Negatiivseid Recency väärtusi: 0
RFM-skoori vahemik: 3 - 15


In [22]:
# Edasijõudnutele: kaalutud RFM-skoor
# Monetary saab suurema kaalu

rfm["RFM_Weighted"] = (
    rfm["R_Score"] * 0.3 +
    rfm["F_Score"] * 0.3 +
    rfm["M_Score"] * 0.4
).round(2)

# Võrdleme tavalist ja kaalutud skoori
display(
    rfm[
        ["customer_id", "RFM_Score", "RFM_Weighted"]
    ].head(10)
)

print(
    "Kaalutud skoori vahemik:",
    rfm["RFM_Weighted"].min(),
    "kuni",
    rfm["RFM_Weighted"].max()
)

,customer_id,RFM_Score,RFM_Weighted
0,2001.0,6,1.9
1,2004.0,9,3.1
2,2005.0,11,3.7
3,2006.0,3,1.0
4,2007.0,7,2.2
5,2008.0,12,4.0
6,2009.0,10,3.5
7,2012.0,8,2.7
8,2013.0,6,2.0
9,2014.0,9,3.1


Kaalutud skoori vahemik: 1.0 kuni 5.0


In [23]:
# Edasijõudnutele: detailsem kliendisegmenteerimine

def detailed_segment(row):
    if row["R_Score"] >= 4 and row["F_Score"] >= 4 and row["M_Score"] >= 4:
        return "VIP Champions"
    elif row["R_Score"] >= 4 and row["F_Score"] >= 3:
        return "Loyal Active"
    elif row["R_Score"] >= 4 and row["F_Score"] <= 2:
        return "New / Promising"
    elif row["R_Score"] <= 2 and row["F_Score"] >= 4:
        return "At Risk High Value"
    elif row["R_Score"] <= 2:
        return "Lost / Inactive"
    else:
        return "Potential"

rfm["Segment_Detailed"] = rfm.apply(detailed_segment, axis=1)

print("Detailsed kliendisegmendid:")
print(rfm["Segment_Detailed"].value_counts())
print("\nKliente kokku:", len(rfm))

Detailsed kliendisegmendid:
Segment_Detailed
Lost / Inactive       734
Potential             508
VIP Champions         411
New / Promising       317
Loyal Active          287
At Risk High Value    281
Name: count, dtype: int64

Kliente kokku: 2538


In [24]:
# Edasijõudnutele: RFM-tulemuste eksport CSV-faili
rfm.to_csv("urbanstyle_rfm_results.csv", index=False)

print("RFM-tulemused salvestatud faili: urbanstyle_rfm_results.csv")
print("Kliente eksporditud:", len(rfm))

RFM-tulemused salvestatud faili: urbanstyle_rfm_results.csv
Kliente eksporditud: 2538


## Roll D - Visualiseerimine

### Knaflic disainipõhimõtete värvipalett (Teal fookus ja neutraalsed hallid)

In [25]:
import plotly.express as px

In [26]:
# Sorteerime segmendid loogilisse järjekorda (VIP - Lost)
segment_order = [
    'VIP Champions', 
    'Loyal Active', 
    'New / Promising', 
    'Potential', 
    'At Risk High Value', 
    'Lost / Inactive'
]

In [27]:
color_palette_segments = {
    'VIP Champions': '#004C4C',       # Tume teal (kõige väärtuslikumad)
    'Loyal Active': '#008080',        # Põhi-teal
    'New / Promising': '#4DADB3',     # Heledam teal
    'Potential': '#A3D1D5',           # Väga hele teal (potentsiaaliga)
    'At Risk High Value': '#969696',  # Neutraalne hall (hoiatus!)
    'Lost / Inactive': '#D3D3D3'      # Helehall (taustale vajuv)
}


GRAY_TEXT = '#333333'
GRAY_LIGHT = '#E0E0E0'
TEAL_MAIN = '#008080'

In [28]:
# Arvutame uute segmentide suurused
df_segments = rfm['Segment_Detailed'].value_counts().reset_index()
df_segments.columns = ['Segment_Detailed', 'Klientide arv']

# Sorteerime uue loogilise järjekorra alusel
df_segments['Segment_Detailed'] = pd.Categorical(
    df_segments['Segment_Detailed'], 
    categories=segment_order, 
    ordered=True
)
df_segments = df_segments.sort_values('Segment_Detailed')

In [29]:
# --- DIAGRAMM 1: Segmentide klientide arv ---
fig1 = px.bar(
    df_segments, 
    x='Segment_Detailed', 
    y='Klientide arv',
    title='<b>Klientide jaotus detailsete segmentide lõikes</b>',
    text='Klientide arv',
    color='Segment_Detailed',
    color_discrete_map=color_palette_segments
)

fig1.update_traces(
    textposition='outside', 
    cliponaxis=False,
    texttemplate='%{text}'
)

fig1.update_layout(
    plot_bgcolor='white', paper_bgcolor='white',
    font=dict(color=GRAY_TEXT, size=12),
    title=dict(font=dict(size=18, color=GRAY_TEXT), x=0.0),
    showlegend=False,
    xaxis=dict(title='', showgrid=False, linecolor=GRAY_LIGHT),
    yaxis=dict(title='', showgrid=False, showticklabels=False),
    margin=dict(l=20, r=20, t=60, b=20)
)
fig1.show()

In [30]:
# DIAGRAMM 2: Hajuvusdiagramm (Recency vs Monetary)
fig2 = px.scatter(
    rfm, 
    x='Recency',          # Muudetud: sinu tabelis on 'Recency'
    y='Monetary',         # Muudetud: sinu tabelis on 'Monetary'
    color='Segment_Detailed', 
    size='Frequency',        # Muudetud: sinu tabelis on 'Frequency'
    title='<b>UrbanStyle Detailsed Kliendisegmendid (RFM)</b>',
    labels={
        'Recency': 'Päevi viimasest ostust',
        'Monetary': 'Kogukulutus (EUR)',
        'Segment_Detailed': 'Detailne segment'
    },
    color_discrete_map=color_palette_segments,
    category_orders={'Segment_Detailed': segment_order}
)

# Knaflicu disain: puhtad peenikesed taustajooned
fig2.update_layout(
    plot_bgcolor='white', 
    paper_bgcolor='white',
    font=dict(color=GRAY_TEXT, size=12),
    title=dict(font=dict(size=18, color=GRAY_TEXT), x=0.0),
    xaxis=dict(showgrid=True, gridcolor=GRAY_LIGHT, gridwidth=0.5, linecolor=GRAY_LIGHT),
    yaxis=dict(showgrid=True, gridcolor=GRAY_LIGHT, gridwidth=0.5, linecolor=GRAY_LIGHT),
    margin=dict(l=20, r=20, t=60, b=20)
)
fig2.show()

In [31]:
#  DIAGRAMM 3: Top 10 VIP klienti kogukulutuse järgi
top_10_vip = rfm[rfm['Segment_Detailed'] == 'VIP Champions'].nlargest(10, 'Monetary').copy()

top_10_vip['Monetary_Label'] = top_10_vip['Monetary'].apply(lambda x: f"{x:,.0f}".replace(",", " ") + " EUR")

fig3 = px.bar(
    top_10_vip,
    x='Monetary',          
    y='customer_id',       
    orientation='h',       
    title='<b>Top 10 VIP klienti kogukulutuse järgi</b>',
    text='Monetary_Label',  # KUVAME OTSE PYTHONIS LOODUD ILUSAT TEKSTISILTI
    labels={'Monetary': 'Kogukulutus (EUR)', 'customer_id': 'Kliendi ID'}
)

# Knaflicu disain: kuvame teksti puhtalt ilma Plotly-poolse ümbervormindamiseta
fig3.update_traces(
    marker_color='#008080',  
    texttemplate='%{text}',  # Võtab otse veeru väärtuse (nt "2 000 EUR")
    textposition='outside',
    cliponaxis=False
)

# Knaflicu disain: puhas ümbrus ja loetavus
fig3.update_layout(
    plot_bgcolor='white', 
    paper_bgcolor='white',
    font=dict(color=GRAY_TEXT, size=12),
    title=dict(font=dict(size=18, color=GRAY_TEXT), x=0.0),
    xaxis=dict(
        showgrid=False, 
        showticklabels=False, 
        zeroline=False, 
        title=''
    ),  
    yaxis=dict(
        type='category',             
        categoryorder='total ascending', 
        showgrid=False, 
        linecolor=GRAY_LIGHT
    ),  
    margin=dict(l=100, r=120, t=60, b=20)  # Paremat marginaali veidi tõstetud, et pikk tekst ära mahuks
)
fig3.show()
   

- VIP Champions - 411 klienti: Meie kõige väärtuslikum tuumik (kõrge ostusagedus, hiljutisus ja kogukulutus). Nad annavad ebaproportsionaalselt suure osa kogukäibest ning moodustavad stabiilse baasi püsiva müügitulu hoidmisel.

- At Risk High Value - 281 klienti: Kliendid, kellel on ajalooliselt kõrge ostusagedus ja Monetary-väärtus, kuid kelle viimasest ostust on möödunud kriitiline aeg (Recency skoor madal). Nad vajavad kohest sekkumist, et ennetada nende liikumist lõplikult passiivsesse faasi.

- Kasvu- ja potentsiaalsed segmendid - 604 klienti: Sisaldab rühmi Loyal Active, New / Promising ja Potential. Nende ostumuster näitab valmisolekut liikuda kõrgemasse väärtustasemesse sobivate tootesoovituste ja stiimulite toel.

- VIP-klubi ja varajane ligipääs kollektsioonidele (VIP Champions):
Tegevus: Loo eksklusiivne, kutsetega UrbanStyle VIP Club.
Fookus: Varajane eel-ligipääs uutele tootekollektsioonidele ning tasuta ekspress-tarned.
Põhjendus: Väldime brändi ja toodete väärtust vähendavaid mass-allahindlusi ning hoiame marginaali, kuna see segment ostab toote väärtuse, mitte allahindlusprotsendi tõttu.

- Sihitud Win-back tootekampaania (At Risk High Value – 281 klienti):
Tegevus: Käivita automatiseeritud korduskampaania, mis põhineb kliendi varasemal ostuajalool.
Fookus: Personaalsed tootesoovitused (ristmüük eelmise ostukategooria põhjal) ja ajaliselt piiratud stiimul.
Põhjendus: Kõrge väärtusega klientide taasaktiviseerimist enne lõplikku passiivseks muutumist.

- Tootehooldus ja Nurture-ahel (New / Promising & Potential – 604 klienti):
Tegevus: 30-päevane e-kirjade tervitusahel esmase ostu järel.
Fookus: Tootematerjalide hooldusjuhendid, brändilugu ja täiendavate toodete soovitused.
Põhjendus: Suurendab korduvostu tõenäosust ja kasvatab kliendi elutsükli väärtust.